# **T4.5.2 Geotagging of texts** [WIP] 

* This workflow is part of the [ATRIUM](https://atrium-research.eu/) project.

* This notebook supports only `.txt` file formats, for other formats please check the available notebooks [here](https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/tree/main/notebooks).

* In this example, we will use a preprocessed `Pausanias.txt` file, downloaded from the [Digital Periegesis](https://www.periegesis.org/en/reports.php?projectid=1).

* In this example we will be using the [MLX framework](https://github.com/ml-explore/mlx-lm) which is for Apple silicon users. If you have a CUDA GPU, please refer to the [llama.cpp](https://github.com/ggml-org/llama.cpp) framework.

#### Requirements

In [ ]:
# Restart the kernel after the first installation
%pip install -r "https://raw.githubusercontent.com/atrium-research/T4.5.2_Geotagging_of_texts/refs/heads/main/notebooks/requirements.txt"

In [ ]:
import os

# Create the folders
# Move your .txt file inside the 'data' folder
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

### **Step 1: Perform Name Entity Recognition (NER) using our pre-trained model**

In this dataset, each paragraph is denoted by 2 newlines `\n\n`.
Change the next cell structure according to your own dataset.

In [ ]:
import spacy
from tqdm import tqdm
import srsly
import os

# Setups NER location path for pre-trained model
ner_location_model_path = "en_deberta_v3_base_ner_historical_location"

# Setup input and output paths
with open("data/pausanias.txt", "r", encoding="utf-8") as f:
    input_data = [{"text":i.replace("\n","").strip()} for i in f.readlines() if i != "\n"]

ner_output_path = "./outputs/ner_data.jsonl"

In [ ]:
def NER(model_path, in_data, entity):
    ner_model = spacy.load(model_path)
    annotated_data = []
    for row in tqdm(in_data, desc=f"NER for {entity}"):
        sent_nlp = ner_model(row["text"])
        ner_spans = [{"start": span.start_char, "end": span.end_char, "label": entity} for span in sent_nlp.ents]
        if "spans" in row:
            row["spans"] += ner_spans
        else:
            row["spans"] = ner_spans

        annotated_data.append(row)

    return annotated_data

In [ ]:
ner_data = NER(ner_location_model_path, input_data, "LOCATION")

srsly.write_jsonl(ner_output_path, ner_data)

### **Step 2: Recontext the NER predictions using a Large Language Model (LLM)**

In [ ]:
# Run this on your termimal
#!mlx_lm.server --model mlx-community/Qwen3.5-9B-OptiQ-4bit --chat-template-args '{"enable_thinking": false}'

In [ ]:
from openai import OpenAI
import srsly
from tqdm import tqdm

input_data = list(srsly.read_jsonl("outputs/ner_data.jsonl"))
llm_output_path = "./outputs/llm_data.jsonl"

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

In [ ]:
# This is the instructions the LLM will use, you can change it according to your task

system_prompt = """
You are an expert historian and archaeologist.

You will receive:
- Mention: a referenced entity (which may be a building, temple, sanctuary, city, region, island, country, monument, harbor, person, or object)
- Context: a short text snippet used only for disambiguation

Task:
Identify exactly what real-world entity the mention refers to using the context, then write a standalone encyclopedic description of that entity.

Important:
- Use the context ONLY to resolve ambiguity.
- Do NOT describe or reference the context, passage, or source text.
- Do NOT explain how the entity appears in the text.
- Do NOT use phrases like:
  "This refers to"
  "The mention refers to"
  "In the text"
- Write as a neutral encyclopedia entry (Wikipedia-style opening).

Output format:
<encyclopedic description>

Rules:
- Write 2–4 concise sentences.
- Focus on identity, location, historical significance, and defining features.
- If multiple candidates exist, choose the most likely one based on context.

Output examples:

Kantharos was the largest harbor of ancient Piraeus and served as the principal commercial and naval port of Athens. It played a central role in Athenian maritime trade and naval operations from the Classical period onward.

Athens is a major city in Greece and one of the most important centers of ancient Greek civilization. It served as a leading political, cultural, and intellectual hub in antiquity.

The Pompeion was a public building in ancient Athens located in the Kerameikos district. It was used for preparing and organizing religious processions, especially the Panathenaic procession.
"""

In [ ]:
def recontext(mention, text, client):
    user_prompt = f'Mention: "{mention}" Context: {text}'
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
    
    response = client.chat.completions.create(
        model = "mlx-community/Qwen3.5-9B-OptiQ-4bit",
        messages=messages,
        max_tokens=81920,
        temperature=1.0,
        top_p=0.95,
        presence_penalty=1.5,
        extra_body={
            "repetition-penalty":1.0,
            "top-k":20,
            "min-p":0.0
        }
    )

    return response.choices[0].message.content

In [ ]:
for row in tqdm(input_data, desc="Text generation"):
    for element in row["spans"]:
        text = row["text"]
        mention = text[element["start"]:element["end"]]
        llm_text = recontext(mention, text, client)
        element.update({"recontext":llm_text})
        
    srsly.write_jsonl(llm_output_path, [row], append=True, append_new_line=False)

### **Step 3: Indexing & fast approximate retrieval**

* The FAISS index was built from [ToposText](https://topostext.org/) database and can be found [here](https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/tag/v.1) along with the metadata file.

In [17]:
import srsly
import faiss
import pickle
from openai import OpenAI
from tqdm import tqdm

In [18]:
input_data = list(srsly.read_jsonl("outputs/llm_data.jsonl"))
llm_output_path = "./outputs/retrieval_data.jsonl"

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

In [19]:
# If you plan to use ToposText as a gazetter then uncomment the 2 lines below:
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext.index -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext_meta.pkl -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/ToposText_gazetteer.json -P ./data/topostext

gazetteer = srsly.read_json("data/topostext/ToposText_gazetteer.json")

index = faiss.read_index("data/topostext/topostext.index")
with open("data/topostext/topostext_meta.pkl", "rb") as f:
    metadata = pickle.load(f)

In [ ]:
for row in tqdm(input_data):
    for mention in row.get("mentions_tagged"):
        
        array = []
        query_text = f"{mention.get("name")}: {mention.get("recontext")}"
        x_query = ollama.embed(model='qwen3-embedding:8b', input=query_text) # The faiss was built with qwen3-embedding:8b
        query_vec = np.array(x_query["embeddings"], dtype=np.float32)
        
        # Comment this if you want to get the top 1 result without running the 4.1 step
        distances, indices = index.search(query_vec, 100)

        # Uncomment this if you want to the the top 1 result without running the 4.1 step
        #distances, indices = index.search(query_vec, 1)

        result_ids = metadata.get("ids")[indices]
        for id,distance in zip(result_ids[0], distances[0]):
            array.append([gazetteer["features"][id].get("@id").split("/")[-1], str(distance)])

        mention.update({"vector_db":array})
    srsly.write_jsonl("files/4_pausanias_faiss.jsonl", [row], append=True, append_new_line=False)

### (OPTIONAL) Step 4.1: Run a Reranker for better results

* We are using the [Qwen3-Reranker-4B](https://huggingface.co/Qwen/Qwen3-Reranker-4B).
* You can use either the [Qwen3-Reranker-8B](https://huggingface.co/Qwen/Qwen3-Reranker-8B) if you have the resources.
* Or the [Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) if you have limited resources.

In [ ]:
rerank_model = "Qwen/Qwen3-Reranker-4B" #change this depending if you want the 4b, the 8b or the 0.6b (e.g. 'Qwen/Qwen3-Reranker-8B', 'Qwen/Qwen3-Reranker-0.6B')

def format_instruction(instruction, query, doc):
    if instruction is None:
        instruction = "Determine whether the Document describes or identifies the same historical place, group, or location referred to in the Query. Answer 'yes' only if it clearly refers to that exact entity, not to a nearby site or people associated with it. Otherwise, answer 'no'."
    output = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(instruction=instruction,query=query, doc=doc)
    return output

def process_inputs(pairs):
    inputs = tokenizer(
        pairs, padding=False, truncation='longest_first',
        return_attention_mask=False, max_length=max_length - len(prefix_tokens) - len(suffix_tokens)
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens
    inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=max_length)
    for key in inputs:
        inputs[key] = inputs[key].to(model.device)
    return inputs

@torch.no_grad()
def compute_logits(inputs, **kwargs):
    batch_scores = model(**inputs).logits[:, -1, :]
    true_vector = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]
    batch_scores = torch.stack([false_vector, true_vector], dim=1)
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
    scores = batch_scores[:, 1].exp().tolist()
    return scores

tokenizer = AutoTokenizer.from_pretrained(rerank_model, padding_side='left')

# FOR GPU - We recommend enabling flash_attention_2 or sdpa for better acceleration and memory saving.
#model = AutoModelForCausalLM.from_pretrained(rerank_model, torch_dtype=torch.float16, attn_implementation="sdpa").to('cuda').eval()

# FOR CPU
model = AutoModelForCausalLM.from_pretrained(rerank_model).eval()

token_false_id = tokenizer.convert_tokens_to_ids("no")
token_true_id = tokenizer.convert_tokens_to_ids("yes")
max_length = 8192

prefix = (
    "<|im_start|>system\n"
    "You are a factual judge. Using the Instruct and the Query, decide whether the Document "
    "clearly describes the same historical place, group, or location. "
    "Answer only with 'yes' or 'no'.\n"
    "<|im_end|>\n"
    "<|im_start|>user\n"
)
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

In [ ]:
topos_data = list(srsly.read_jsonl("files/4_pausanias_faiss.jsonl"))
data = srsly.read_json("data/ToposText_gazetteer.json")
data = data["features"]

task = "Determine whether the Document describes or identifies the same historical place, group, or location referred to in the Query. Answer 'yes' only if it clearly refers to that exact entity, not to a nearby site or people associated with it. Otherwise, answer 'no'."

In [ ]:
for i in tqdm(topos_data):
    for mention in i.get("mentions_tagged"):
        topos_text_ids = [i[0] for i in mention.get("vector_db")][:30]
        hash_map = []
        final_list = []

        for id in topos_text_ids:
            for d in data:
                if d.get("@id").split("/")[-1] == id:
                    title = d.get("properties").get("title")
                    description = d.get("properties").get("description")
                    hash_map.append({id:f"{title}: {description}"})
                    break

        for element in hash_map:
            query = f"The mention '{mention.get("name")}' appears in the context: {mention.get("recontext")}"
            pairs = [format_instruction(task, query, list(element.values())[0])]

            inputs = process_inputs(pairs)
            scores = compute_logits(inputs)

            final_list.append({list(element.keys())[0]:scores[0]})
        
        final_list = sorted(final_list, key=lambda x: list(x.values())[0], reverse=True)[0]
        mention.update({"reranker":[final_list]})
    
    srsly.write_jsonl("files/4_1_pausanias_rerank.jsonl", [i], append=True, append_new_line=False)

### Step 5: Create the input for the [Recogito Studio](https://recogitostudio.org/)

In [ ]:
# We assume that we executed the 4.1 optional step, comment this if you didn't execute it
data = list(srsly.read_jsonl("files/4_1_pausanias_rerank.jsonl"))

# Uncomment this if you didn't execute the 4.1 optional step
#data = list(srsly.read_jsonl("files/4_pausanias_faiss.jsonl"))

In [ ]:
NS_TEI = "http://www.tei-c.org/ns/1.0"
NS_XML = "http://www.w3.org/XML/1998/namespace"
NSMAP = {None: NS_TEI}

In [ ]:
counter = 1
uid_counter = 0
chapter = 0
current_book = None
current_chapter = None

In [ ]:
# If you want each chapter-book pair to be a different XML run this cell
for i in data:
    book = i.get("book")
    chapter = i.get("chapter")

    if current_chapter is not None and chapter != current_chapter:
        tree = etree.ElementTree(tei)
        tree.write(
            f"books_chapters/pausanias_book_{current_book}_chapter_{current_chapter}.xml",
            xml_declaration=True,
            encoding="utf-8",
            pretty_print=True
        )

        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1

        head = etree.SubElement(body, "head")
        head.text = f"Book {book}, Chapter {chapter}"

    if current_chapter is None:
        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1

        head = etree.SubElement(body, "head")
        head.text = f"Book {book}, Chapter {chapter}"

    current_book = book
    current_chapter = chapter

    p = etree.SubElement(body, "p")
    p.text = i.get("text")

    for mention in i.get("mentions_tagged"):
        annotation = etree.SubElement(listannotation, "annotation", target=f"/TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("start"))} /TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("end"))}")
        annotation.set(f"{{{NS_XML}}}id", f"UID-FAKE-{uid_counter}")

        # Comment this if you didn't run the optional step
        topos_id = list(mention.get("reranker")[0].keys())[0]
        
        # Uncomment this if you didn't run the optional step
        #topos_id = mention.get("vector_db")[0][0]
        
        rs = etree.SubElement(annotation, "rs", ana=f"https://topostext.org/place/{topos_id}")
        uid_counter += 1

    counter += 1

tree = etree.ElementTree(tei)
tree.write(
    f"books_chapters/pausanias_book_{current_book}_chapter_{current_chapter}.xml",
    xml_declaration=True,
    encoding="utf-8",
    pretty_print=True
)

In [ ]:
# If you want each book to be a different XML run this
for i in data:
    if current_book is not None and i.get("book") != current_book:
        tree = etree.ElementTree(tei)
        tree.write(
            f"books/pausanias_book_{current_book}.xml",
            xml_declaration=True,
            encoding="utf-8",
            pretty_print=True
        )

        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1
        chapter = 0 

        head = etree.SubElement(body, "head")
        head.text = f"Book {i.get("book")}"

    current_book = i.get("book")

    if i.get("chapter") != chapter:
        head = etree.SubElement(body, "head")
        head.text = f"Chapter {i.get("chapter")}"
        chapter = i.get("chapter")

    p = etree.SubElement(body, "p")
    p.text = i.get("text")
    for mention in i.get("mentions_tagged"):
        annotation = etree.SubElement(listannotation, "annotation", target=f"/TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("start"))} /TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("end"))}")
        annotation.set(f"{{{NS_XML}}}id", f"UID-FAKE-{uid_counter}")

        # Comment this if you didn't run the optional step
        topos_id = list(mention.get("reranker")[0].keys())[0]
        
        # Uncomment this if you didn't run the optional step
        #topos_id = mention.get("vector_db")[0][0]

        rs = etree.SubElement(annotation, "rs", ana=f"https://topostext.org/place/{topos_id}")
        uid_counter += 1

    counter += 1

tree = etree.ElementTree(tei)
tree.write(
    f"books/pausanias_book_{current_book}.xml",
    xml_declaration=True,
    encoding="utf-8",
    pretty_print=True
)